# 07 · O modelo com maior F1 muda em outro período?

Análise complementar ao TCC sobre HAI 21.03. As células abaixo já foram executadas; você pode abrir este notebook no VS Code e ler os resultados sem repetir o treino. Para refazer a análise, consulte a última célula. Ao contrário das três sementes da subamostragem, aqui muda **o período de teste**.

In [1]:
from pathlib import Path
import pandas as pd

# Se a pasta extraída estiver em outro local, informe o caminho em PASTA.
PASTA = Path.cwd()
if not (PASTA / "resultados" / "metricas_por_periodo.csv").is_file():
    PASTA = PASTA / "HAI_Avaliacao_Temporal"
r = PASTA / "resultados"
met = pd.read_csv(r / "metricas_por_periodo.csv", dtype={"versao":str})
periodos = pd.read_csv(r / "inventario_dos_periodos.csv", dtype={"versao":str})
periodos[["versao", "arquivo", "inicio", "fim", "normais", "registros_ataque"]].drop_duplicates().reset_index(drop=True)

,versao,arquivo,inicio,fim,normais,registros_ataque
0,21.03,test3,2020-07-13 00:00:00,2020-07-14 06:00:00,106466,1535
1,21.03,test4,2020-07-28 12:00:00,2020-07-28 23:00:00,38444,1157
2,21.03,test5,2020-07-30 10:50:00,2020-07-31 12:30:00,90224,2177
3,22.04,test1,2021-07-10 00:00:01,2021-07-11 00:00:00,85515,885
4,22.04,test2,2021-07-13 16:00:01,2021-07-14 15:00:00,79919,2881
5,22.04,test3,2021-07-14 17:40:01,2021-07-15 11:00:00,58559,3841
6,22.04,test4,2021-07-15 12:00:01,2021-07-17 00:00:00,125177,4423


## 1. Três testes dentro da versão 21.03

- **J1:** aprende em test1, escolhe o limiar em test2, avalia em test3.
- **J2:** aprende em test1–2, escolhe o limiar em test3, avalia em test4.
- **J3:** aprende em test1–3, escolhe o limiar em test4, avalia em test5.

O modelo é refeito em cada rodada. Os três períodos de teste não se sobrepõem, mas os conjuntos de treino compartilham dados. Um F1 maior indica um melhor equilíbrio entre precisão e recall por registro.

In [2]:
internas = met[met["rodada"].isin(["J1", "J2", "J3"])]
f1 = internas.pivot(index="modelo", columns="arquivo", values="f1")
f1.round(4)

arquivo,test3,test4,test5
modelo,,,
Ensemble OU,0.0606,0.4363,0.5534
IF com PCA,0.0606,0.3157,0.4611
IF sem PCA,0.0943,0.1520,0.2203
RF com pesos,0.0564,0.4652,0.1871
Regressão Logística,0.0437,0.3300,0.5672


In [3]:
melhores = internas.loc[internas.groupby("arquivo")["f1"].idxmax()]
melhores[["arquivo", "modelo", "f1", "FP", "episodios_sinalizados", "episodios_total"]].round(4).reset_index(drop=True)

,arquivo,modelo,f1,FP,episodios_sinalizados,episodios_total
0,test3,IF sem PCA,0.0943,14183,8,8
1,test4,RF com pesos,0.4652,331,4,5
2,test5,Regressão Logística,0.5672,642,12,12


O primeiro lugar mudou nas três rodadas. Interprete junto com os números absolutos: o maior F1 em test3 foi só 0,0943. Na tabela principal do artigo, a execução original da Logística em test5 produziu F1 de 0,5671. Aqui a reexecução usa scikit-learn 1.8.0.

## 2. Aplicação na coleta HAI 22.04

Quatro arquivos rotulados; 58 episódios. Os métodos foram treinados e calibrados apenas no HAI 21.03 com as 52 entradas presentes nas duas versões. As métricas abaixo são calculadas pela soma das contagens dos quatro testes.

In [4]:
ext = pd.read_csv(r / "externo_agregado.csv")
ext[["modelo", "f1", "precisao", "recall", "FP", "taxa_fp", "episodios_sinalizados", "episodios_total"]].round(4)

,modelo,f1,precisao,recall,FP,taxa_fp,episodios_sinalizados,episodios_total
0,RF com pesos,0.0508,0.0277,0.3060,129118,0.3698,47,58
1,IF sem PCA,0.0645,0.0333,1.0000,349170,1.0000,58,58
2,IF com PCA,0.0645,0.0333,0.9997,348953,0.9994,58,58
3,Regressão Logística,0.0643,0.0332,0.9981,349170,1.0000,58,58
4,Ensemble OU,0.0645,0.0333,1.0000,349170,1.0000,58,58


In [5]:
ext_f1 = met[met["rodada"].eq("EXT")].pivot(index="modelo", columns=["versao", "arquivo"], values="f1")
ext_f1.round(4)

versao                21.03   22.04                        
arquivo               test5   test1   test2   test3   test4
modelo                                                     
Ensemble OU          0.5410  0.0203  0.0672  0.1160  0.0660
IF com PCA           0.4025  0.0203  0.0672  0.1160  0.0660
IF sem PCA           0.1971  0.0203  0.0672  0.1160  0.0660
RF com pesos         0.1779  0.0175  0.0490  0.0684  0.0595
Regressão Logística  0.6095  0.0203  0.0672  0.1153  0.0660

O IF sem PCA e o Ensemble OU sinalizaram todos os registros externos. Eles cobriram todos os episódios, mas também todos os segundos normais. A Logística sinalizou todos os registros normais e deixou 23 registros de ataque sem alerta.

## 3. O que mudou nos sinais?

Depois de avaliar, comparamos os registros normais de cada coleta com a faixa (mínimo e máximo) dos normais do treino. Esta etapa não altera os modelos ou limiares.

In [6]:
sinais = pd.read_csv(r / "diagnostico_faixas_sinais.csv")
fora = sinais[(sinais["periodo"] == "externo_22.04") & sinais["mediana_fora_do_intervalo"]]
fora[["sinal", "min_treino_normal", "max_treino_normal", "mediana_teste_normal", "fracao_fora"]].round(4).reset_index(drop=True)

,sinal,min_treino_normal,max_treino_normal,mediana_teste_normal,fracao_fora
0,P1_B2004,0.0301,0.1018,0.1269,0.7431
1,P1_B3004,381.9789,435.5023,456.4247,0.6511
2,P1_B4002,31.3790,33.6555,25.6872,1.0000
3,P1_B4022,34.3510,38.3544,29.4885,1.0000
4,P1_TIT01,34.6130,36.9476,30.1727,1.0000
5,P1_TIT02,34.9792,40.4114,42.0746,0.6404
6,P2_SIT01,0.0000,887.0000,908.0000,0.9799
7,P4_ST_TT01,27531.0000,27627.0000,27090.0000,1.0000


Oito medianas estavam fora da faixa do treino. Isso mostra uma diferença nos valores observados. Os dados, sozinhos, não dizem se a causa é escala, instrumentação ou condição de operação.

## 4. Onde estão os limiares?

In [7]:
cal = pd.read_csv(r / "limiares_e_validacao.csv")
cal.pivot(index="modelo", columns="rodada", values="limiar").round(8)

rodada,EXT,J1,J2,J3
modelo,,,,
Ensemble OU,NaN,NaN,NaN,NaN
IF com PCA,0.566358,0.581258,0.675114,0.567999
IF sem PCA,0.547731,0.560176,0.574957,0.548756
RF com pesos,0.200000,0.010000,0.190000,0.190000
Regressão Logística,0.190801,0.999916,0.999293,0.152612


O ensemble não tem limiar próprio: ele usa a regra OU sobre dois alertas já decididos. Os limiares com a precisão usada no teste estão nos arquivos `protocolo_*.json`.

## Para treinar novamente

Veja o **LEIA-ME.md** para a revisão dos dados, versões e comandos. As métricas salvas são suficientes para discutir os resultados; não é necessário refazer os treinos. Para reproduzir, rode `comparar_periodos.py`, `resumir.py` e `conferir_sinais.py` como indicado no LEIA-ME, sem sobrescrever a pasta original de resultados.